# Install Required Libraries

In [ ]:
!pip install -q --upgrade \
    youtube-transcript-api \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-groq \
    sentence-transformers \
    faiss-cpu

# System Imports & Environment Setup




In [ ]:
import os
import re
from google.colab import userdata
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("Secrets configured: GROQ_API_KEY loaded successfully.")
except Exception:
    print("Warning: GROQ_API_KEY not found in Colab Secrets.")
    print("Add 'GROQ_API_KEY' under the 🔑 (Secrets) tab on the left sidebar.")

# Extract YouTube Transcript with Timestamps

In [ ]:
def get_youtube_transcript(video_url):
    """Extracts transcript and timestamps from a YouTube URL."""
    video_id_match = re.search(r"(?:v=|\/)([0-9A-Za-z_-]{11}).*", video_url)
    if not video_id_match:
        print("Error: Could not extract video ID. Please check the URL.")
        return None

    video_id = video_id_match.group(1)
    print(f"Extracting transcript for video ID: {video_id}...")

    try:
        yt_api = YouTubeTranscriptApi()
        fetched_transcript = yt_api.fetch(video_id)

        transcript_dicts = fetched_transcript.to_raw_data()

        print(f"Success! Extracted {len(transcript_dicts)} text segments.")
        return transcript_dicts
    except Exception as e:
        print(f"Error fetching transcript: {e}")
        return None

user_url = ""
while not user_url:
    user_url = input("Enter a YouTube URL to analyze: ").strip()
    if not user_url:
        print("Error: URL cannot be empty. Please enter a valid YouTube link.")

raw_transcript = get_youtube_transcript(user_url)

# Display a sneak peek of the first 3 segments to verify the structure
if raw_transcript:
    for segment in raw_transcript[:3]:
        print(f"[{segment['start']:.2f}s]: {segment['text']}")

# Smart Semantic Chunking with Timestamps


In [ ]:
def create_timestamped_chunks(transcript_dicts, target_chunk_size=1000):
    """
    Groups tiny video snippets into larger, semantically coherent LangChain Documents.
    Preserves the start time of the first snippet in the chunk for exact video jumping.
    """
    if not transcript_dicts:
        print("Error: No transcript data available to chunk.")
        return []

    print("Formatting and chunking transcript...")

    documents = []
    current_text = ""
    current_start = None

    for segment in transcript_dicts:
        if current_start is None:
            current_start = segment['start']

        current_text += segment['text'] + " "

        if len(current_text) >= target_chunk_size:
            doc = Document(
                page_content=current_text.strip(),
                metadata={"start_time": current_start}
            )
            documents.append(doc)
            current_text = ""
            current_start = None

    if current_text:
        documents.append(Document(
            page_content=current_text.strip(),
            metadata={"start_time": current_start}
        ))

    print(f"Success! Compressed {len(transcript_dicts)} tiny snippets into {len(documents)} semantic chunks.")
    return documents

chunked_docs = create_timestamped_chunks(raw_transcript)


# Vector Embeddings and FAISS Vector Database

In [ ]:
if not chunked_docs:
    print("Error: No chunked documents available to embed. Please check the video URL.")
else:
    print("Downloading embedding model (this might take a moment)...")

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    print("Converting chunks to vector embeddings and building FAISS database...")
    # Create the FAISS index from our chunked documents
    vectorstore = FAISS.from_documents(chunked_docs, embeddings)

    print(f"Success! Vector database created with {vectorstore.index.ntotal} embeddings.")

# RAG Pipeline with Temporal Prompt Engineering

In [ ]:
# Initialize Groq LLM
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.2
)

# Prompt setup
system_prompt = (
    "You are an intelligent educational assistant named ChronoSeek. "
    "Use the following pieces of retrieved video transcript to answer the user's question. "
    "For EVERY claim or piece of information you provide, you MUST cite the exact timestamp "
    "provided in the context segment. Format your citations like this: [142.50s]. "
    "If the answer is not in the context, state that you do not know. Do not hallucinate."
    "\n\n"
    "Context (with timestamps):\n"
    "{context}"
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

def ask_chronoseek(query):
    docs = vectorstore.similarity_search(query, k=3)
    formatted_context = ""
    for doc in docs:
        formatted_context += f"[Timestamp: {doc.metadata['start_time']:.2f}s]\n{doc.page_content}\n\n"

    chain = prompt_template | llm
    response = chain.invoke({
        "context": formatted_context,
        "input": query
    })
    return response.content

# Run ChronoSeek Chat Interface

In [ ]:
print("Type your question below and press Enter. (Type 'exit' or 'q' to quit.) \n")

while True:
    # Prompt the user for a query at runtime
    user_query = input("Ask a question about the video: ")

    if user_query.lower().strip() in ['exit', 'quit', 'q', 'nothing']:
        print("Exiting.. Goodbye!")
        break

    # Skip empty inputs
    if not user_query.strip():
        continue

    print("Thinking...")
    answer = ask_chronoseek(user_query)

    print("\n--- RESPONSE ---")
    print(answer)
    print("-" * 50 + "\n")